In [63]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import onnxruntime as rt
import onnx
from skl2onnx.common.data_types import FloatTensorType
from skl2onnx import to_onnx
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from skl2onnx import convert_sklearn

In [64]:
# Let's load the dataset
data = pd.read_csv('data/synth_data_for_training.csv')

# Let's specify the features and the target
y = data['checked']
X = data.drop(['checked'], axis=1)
X = X.astype(np.float32)

# Let's split the dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [65]:
# Select data based on variance (not the final version yet, for now just for testing)
selector = VarianceThreshold()

In [66]:
# Define a gradient boosting classifier
classifier = GradientBoostingClassifier(n_estimators=100, learning_rate=1.0, max_depth=1, random_state=0)

In [67]:
# Create a pipeline object with our selector and classifier
# NOTE: You can create custom pipeline objects but they must be registered to onnx or it will not recognise them
# Because of this we recommend using the onnx known objects as defined in the documentation
pipeline = Pipeline(steps=[('feature selection', selector), ('classification', classifier)])

In [68]:
# Let's train a simple model
pipeline.fit(X_train, y_train)

# Let's evaluate the model
y_pred = pipeline.predict(X_test)
original_accuracy = accuracy_score(y_test, y_pred)
print('Accuracy of the original model: ', original_accuracy)

Accuracy of the original model:  0.9456040480708412


In [69]:
# Let's convert the model to ONNX
onnx_model = convert_sklearn(
    pipeline, initial_types=[('X', FloatTensorType((None, X.shape[1])))],
    target_opset=12)

# Let's check the accuracy of the converted model
sess = rt.InferenceSession(onnx_model.SerializeToString())
y_pred_onnx =  sess.run(None, {'X': X_test.values.astype(np.float32)})

accuracy_onnx_model = accuracy_score(y_test, y_pred_onnx[0])
print('Accuracy of the ONNX model: ', accuracy_onnx_model)

Accuracy of the ONNX model:  0.9456040480708412


In [70]:
# Let's save the model
onnx.save(onnx_model, "model/gboost.onnx")

# Let's load the model
new_session = rt.InferenceSession("model/gboost.onnx")

# Let's predict the target
y_pred_onnx2 =  new_session.run(None, {'X': X_test.values.astype(np.float32)})

accuracy_onnx_model = accuracy_score(y_test, y_pred_onnx2[0])
print('Accuracy of the ONNX model: ', accuracy_onnx_model)


Accuracy of the ONNX model:  0.9456040480708412


## Partitioning based on gender

In [48]:
# Evaluate model performance per gender (and handle class imbalance)
from sklearn.metrics import precision_recall_fscore_support, classification_report, balanced_accuracy_score

col = 'persoon_geslacht_vrouw'  # change if your column name differs
input_name = new_session.get_inputs()[0].name

overall = {'sklearn': {}, 'onnx': {}}

for val in sorted(X_test[col].unique()):
    idx = X_test[X_test[col] == val].index
    X_sub = X_test.loc[idx]
    y_sub = y_test.loc[idx]
    n = len(X_sub)
    print('#' * 60)
    print(f'Group {col}={val}: n={n}')
    if n == 0:
        print('  no samples for this group')
        continue

    # ONNX prediction (if session is available)
    
    y_pred_onnx = new_session.run(None, {input_name: X_sub.values.astype(np.float32)})[0]
    w_prec_o, w_rec_o, w_f1_o, _ = precision_recall_fscore_support(y_sub, y_pred_onnx, average='weighted', zero_division=0)
    bal_acc_o = balanced_accuracy_score(y_sub, y_pred_onnx)
    print('ONNX model metrics:')
    print(f'  Weighted precision={w_prec_o:.4f}, recall={w_rec_o:.4f}, f1={w_f1_o:.4f}, balanced_acc={bal_acc_o:.4f}')
    print('  Classification report:\n', classification_report(y_sub, y_pred_onnx, zero_division=0))
    
    


############################################################
Group persoon_geslacht_vrouw=0.0: n=1664
ONNX model metrics:
  Weighted precision=0.9356, recall=0.9399, f1=0.9367, balanced_acc=0.7869
  Classification report:
               precision    recall  f1-score   support

           0       0.96      0.98      0.97      1496
           1       0.76      0.60      0.67       168

    accuracy                           0.94      1664
   macro avg       0.86      0.79      0.82      1664
weighted avg       0.94      0.94      0.94      1664

############################################################
Group persoon_geslacht_vrouw=1.0: n=1498
ONNX model metrics:
  Weighted precision=0.9489, recall=0.9519, f1=0.9495, balanced_acc=0.8108
  Classification report:
               precision    recall  f1-score   support

           0       0.96      0.98      0.97      1360
           1       0.80      0.64      0.71       138

    accuracy                           0.95      1498
   macro 

In [53]:
## To check about statistical significance as well



# # Summary comparison
# print('#' * 60)
# print('SUMMARY: Performance Comparison Across Gender')
# print('#' * 60)
# results_df = pd.DataFrame(results)
# print(results_df[['gender', 'n_samples', 'balanced_accuracy', 'weighted_f1']])

# # Statistical significance test (optional)
# print('\nPerformance Differences:')
# if len(results_df) == 2:
#     diff_bal_acc = abs(results_df.iloc[0]['balanced_accuracy'] - results_df.iloc[1]['balanced_accuracy'])
#     diff_f1 = abs(results_df.iloc[0]['weighted_f1'] - results_df.iloc[1]['weighted_f1'])
#     print(f'Absolute difference in Balanced Accuracy: {diff_bal_acc:.4f}')
#     print(f'Absolute difference in Weighted F1: {diff_f1:.4f}')
    
#     # Rule of thumb: differences > 0.05 may be concerning for fairness
#     if diff_bal_acc > 0.05 or diff_f1 > 0.05:
#         print('\n⚠️  WARNING: Substantial performance difference detected between groups!')
#         print('   This may indicate potential bias in the model.')
#     else:
#         print('\n✓ Performance appears relatively consistent across groups.')
        


Partition based on language understanding

In [61]:
# Evaluate model performance per gender (and handle class imbalance)
from sklearn.metrics import precision_recall_fscore_support, classification_report, balanced_accuracy_score

col = 'persoonlijke_eigenschappen_taaleis_voldaan'  # change if your column name differs
input_name = new_session.get_inputs()[0].name

overall = {'sklearn': {}, 'onnx': {}}

for val in sorted(X_test[col].unique()):
    idx = X_test[X_test[col] == val].index
    X_sub = X_test.loc[idx]
    y_sub = y_test.loc[idx]
    n = len(X_sub)
    print('#' * 60)
    print(f'Group {col}={val}: n={n}')
    if n == 0:
        print('  no samples for this group')
        continue

    # ONNX prediction (if session is available)
    
    y_pred_onnx = new_session.run(None, {input_name: X_sub.values.astype(np.float32)})[0]
    w_prec_o, w_rec_o, w_f1_o, _ = precision_recall_fscore_support(y_sub, y_pred_onnx, average='weighted', zero_division=0)
    bal_acc_o = balanced_accuracy_score(y_sub, y_pred_onnx)
    print('ONNX model metrics:')
    print(f'  Weighted precision={w_prec_o:.4f}, recall={w_rec_o:.4f}, f1={w_f1_o:.4f}, balanced_acc={bal_acc_o:.4f}')
    print('  Classification report:\n', classification_report(y_sub, y_pred_onnx, zero_division=0))

############################################################
Group persoonlijke_eigenschappen_taaleis_voldaan=0.0: n=1275
ONNX model metrics:
  Weighted precision=0.9209, recall=0.9239, f1=0.9220, balanced_acc=0.8127
  Classification report:
               precision    recall  f1-score   support

           0       0.95      0.96      0.96      1104
           1       0.74      0.66      0.70       171

    accuracy                           0.92      1275
   macro avg       0.85      0.81      0.83      1275
weighted avg       0.92      0.92      0.92      1275

############################################################
Group persoonlijke_eigenschappen_taaleis_voldaan=1.0: n=1749
ONNX model metrics:
  Weighted precision=0.9614, recall=0.9646, f1=0.9609, balanced_acc=0.7627
  Classification report:
               precision    recall  f1-score   support

           0       0.97      0.99      0.98      1642
           1       0.83      0.53      0.65       107

    accuracy           

https://docs.google.com/document/d/16xPqnaMLG6FbFvz_SE2QchHNcfVXV3HQvNzQLXHMxvg/edit?usp=sharing

In [72]:
import pandas as pd
import numpy as np
import onnxruntime as rt
from sklearn.metrics import confusion_matrix

class FairnessTesterONNX:
    def __init__(self, session, X_test, y_test, input_name='X'):
        """
        session: The loaded rt.InferenceSession
        X_test: The Pandas DataFrame containing test features
        y_test: The Pandas Series or array containing true labels
        input_name: The name of the input node for the ONNX model (usually 'X' or 'float_input')
        """
        self.session = session
        self.X_test = X_test.copy() # Keep as DF to manipulate columns easier
        self.y_test = np.array(y_test)
        self.input_name = input_name

    def _get_predictions(self, df_input):
        """
        Helper function to handle ONNX strict input requirements.
        Converts DataFrame -> Float32 Numpy Array -> ONNX Format
        """
        # Convert dataframe to float32 numpy array
        input_data = df_input.values.astype(np.float32)
        
        # Run inference
        # returns a list: [labels, probabilities]. We take [0] for labels.
        onnx_outputs = self.session.run(None, {self.input_name: input_data})
        return onnx_outputs[0]

    def get_fpr(self, y_true, y_pred):
        """Calculate False Positive Rate safely"""
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        if (fp + tn) == 0: return 0.0
        return fp / (fp + tn)

    # ---------------------------------------------------------
    # TEST 1: Partitioning (Intersectional / Subgroup Fairness)
    # ---------------------------------------------------------
    def test_partition(self, condition_mask, description):
        """
        Calculates FPR for a subgroup vs the rest of the population.
        condition_mask: A boolean Series (True for the subgroup)
        """
        print(f"\n🔹 TESTING PARTITION: {description}")
        
        # Split Data
        subgroup_X = self.X_test[condition_mask]
        subgroup_y = self.y_test[condition_mask]
        
        rest_X = self.X_test[~condition_mask]
        rest_y = self.y_test[~condition_mask]
        
        if len(subgroup_X) == 0:
            print("Skipping: No data found for this subgroup.")
            return

        # Get Predictions
        pred_subgroup = self._get_predictions(subgroup_X)
        pred_rest = self._get_predictions(rest_X)
        
        # Calculate FPR
        fpr_subgroup = self.get_fpr(subgroup_y, pred_subgroup)
        fpr_rest = self.get_fpr(rest_y, pred_rest)
        
        print(f"   Subgroup FPR: {fpr_subgroup:.2%}")
        print(f"   Rest of Pop FPR: {fpr_rest:.2%}")
        print(f"   Difference:   {(fpr_subgroup - fpr_rest):.2%}")
        
        if (fpr_subgroup - fpr_rest) > 0.10: # Threshold of 10%
            print("   ⚠️ FAIL: Significant bias detected against this subgroup.")
        else:
            print("   ✅ PASS: Bias is within acceptable limits.")

    # ---------------------------------------------------------
    # TEST 2: Metamorphic (Counterfactual / Invariance)
    # ---------------------------------------------------------
    def test_metamorphic(self, col_name, value_from, value_to):
        """
        Checks if changing ONE column flips the prediction from 0 (Safe) to 1 (Fraud).
        """
        print(f"\n🔹 TESTING METAMORPHIC: Swapping '{col_name}' from {value_from} -> {value_to}")
        
        # 1. Get Baseline Predictions
        original_preds = self._get_predictions(self.X_test)
        
        # 2. Find Candidates: People who are currently 'value_from' AND predicted as Safe (0)
        # We assume numerical comparisons (allow wiggle room for float/int issues)
        candidate_mask = (np.isclose(self.X_test[col_name], value_from)) & (original_preds == 0)
        
        candidates = self.X_test[candidate_mask].copy()
        
        if len(candidates) == 0:
            print("   No suitable candidates found (Safe people with original value).")
            return

        # 3. Apply the Transformation (The "Metamorphic Relation")
        candidates_modified = candidates.copy()
        candidates_modified[col_name] = value_to
        
        # 4. Predict on Modified Data
        new_preds = self._get_predictions(candidates_modified)
        
        # 5. Check for Flips
        flips = np.sum(new_preds == 1)
        total = len(new_preds)
        
        print(f"   Tested on {total} individuals.")
        print(f"   Predictions flipped to FRAUD: {flips}")
        
        if flips > 0:
            print(f"   ⚠️ WARNING: {flips} people were flagged purely because of this variable change.")
        else:
            print(f"   ✅ PASS: Model remained stable.")

In [183]:
# 1. Initialize the Tester
# Note: 'X' is the input name from your snippet {'X': ...}
tester = FairnessTesterONNX(new_session, X_test, y_test, input_name='X')

# --- TEST CASE 1: PARTITIONING (Intersectional) ---
# Example: Are Young people (Age < 30) having a partner for less than 206 days flagged more ?

mask_young_partner = (X_test['persoon_leeftijd_bij_onderzoek'] < 30) & (X_test['relatie_partner_totaal_dagen_partner'] < 206)

tester.test_partition(
    condition_mask=mask_young_partner, 
    description="young and marriage duration"
)

# --- TEST CASE 2: PARTITIONING (Intersectional) ---
# Example: Are unmarried males who don't show drive and ambition, who have more than 2 children flagged more ?

single_males_no_drive_children = (X_test['persoon_geslacht_vrouw'] == 0) & (X_test['competentie_gedrevenheid_en_ambitie_tonen'] == 0) & (X_test['relatie_kind_huidige_aantal']>2) & (X_test['relatie_partner_huidige_partner___partner__gehuwd_']==0)

tester.test_partition(
    condition_mask=single_males_no_drive_children, 
    description="language and competency"
)

# --- TEST CASE 3: PARTITIONING (Intersectional) ---
# Example: Are unmarried males who don't show drive and ambition, who have more than 2 children flagged more ?

# female_with_children = (X_test['persoon_geslacht_vrouw'] == 0) & (X_test['relatie_partner_huidige_partner___partner__gehuwd_']==0) & (X_test['relatie_kind_huidige_aantal']>2)

# tester.test_partition(
#     condition_mask=female_with_children, 
#     description="single female and having children"
# )

female_with_financial_problems = (X_test['persoon_geslacht_vrouw'] == 1) & (X_test['belemmering_financiele_problemen']==1) & (X_test['relatie_kind_huidige_aantal']>2)

tester.test_partition(
    condition_mask=female_with_financial_problems, 
    description="single female and having financial problems"
)


# ---  METAMORPHIC (Robustness) ---
# Changing 'Gender' from male to female
# Does changing from male to female make them a fraudster?

tester.test_metamorphic(
    col_name='persoon_geslacht_vrouw',  # Replace with actual column name
    value_from=0,               # Original value (e.g., Speaks Dutch)
    value_to=1                  # New value (e.g., Non-Dutch)
)

# Example 2: Does having slightly less income flip the prediction?
tester.test_metamorphic(
    col_name='pla_historie_ontwikkeling',  # Replace with actual column name
    value_from=1,               # Original value (e.g., Speaks Dutch)
    value_to=0                  # New value (e.g., Non-Dutch)
)


# for col in X_test.columns:
    # tester.test_metamorphic(
    #     col_name=col, 
    #     value_from=1, 
    #     value_to=0
    # )


🔹 TESTING PARTITION: young and marriage duration
   Subgroup FPR: 16.67%
   Rest of Pop FPR: 1.86%
   Difference:   14.81%
   ⚠️ FAIL: Significant bias detected against this subgroup.

🔹 TESTING PARTITION: language and competency
   Subgroup FPR: 14.29%
   Rest of Pop FPR: 1.86%
   Difference:   12.43%
   ⚠️ FAIL: Significant bias detected against this subgroup.

🔹 TESTING PARTITION: single female and having financial problems
   Subgroup FPR: 0.00%
   Rest of Pop FPR: 1.90%
   Difference:   -1.90%
   ✅ PASS: Bias is within acceptable limits.

🔹 TESTING METAMORPHIC: Swapping 'persoon_geslacht_vrouw' from 0 -> 1
   Tested on 1532 individuals.
   Predictions flipped to FRAUD: 35
   ⚠️ WARNING: 35 people were flagged purely because of this variable change.

🔹 TESTING METAMORPHIC: Swapping 'pla_historie_ontwikkeling' from 1 -> 0
   Tested on 1901 individuals.
   Predictions flipped to FRAUD: 104
   ⚠️ WARNING: 104 people were flagged purely because of this variable change.


In [176]:
X_test['pla_historie_ontwikkeling']

1688     1.0
7251     0.0
5329     1.0
1697     1.0
8200     1.0
        ... 
5646     1.0
10391    0.0
4083     1.0
4023     1.0
2759     0.0
Name: pla_historie_ontwikkeling, Length: 3162, dtype: float32

In [90]:
X_test['persoonlijke_eigenschappen_taaleis_voldaan'].unique()

array([1., 0., 2.], dtype=float32)

In [177]:
X_test['pla_historie_ontwikkeling'].unique()

array([1., 0.], dtype=float32)